see source at [mmaction2/demo](https://github.com/open-mmlab/mmaction2/blob/main/demo/demo.py#L34)

In [1]:
%matplotlib inline

from __future__ import annotations

import os
import argparse
from typing import Optional, Tuple

import torch
import torch.nn as nn
import numpy as np

from mmengine import Config, DictAction
from mmaction.apis import inference_recognizer, init_recognizer

import sys

sys.path.append('../../../')

%load_ext autoreload
%autoreload 2
    
from computer_vision.slowfast.mmaction.utils import SampleList
from computer_vision.slowfast.mmaction.models.recognizers.recognizer3d import Recognizer3D

In [7]:
def parse_args(argument:list[str]|None=None):
    """ Parsing input arguments
    Args:
        argument (list[str], optional): List of input arguments, each element is for one argument
    Returns:
        (argparse.Namespace): Argument namespace object
    """
    parser=argparse.ArgumentParser(description='MMAction2 demo')
    parser.add_argument('config', help='test config file')
    parser.add_argument('checkpoint', help='checkpoint file/url')
    parser.add_argument('video', help='video file/url or rawframe directory')
    parser.add_argument('label', help='label file')
    parser.add_argument('--device', type=str, default='cpu', choices=['cuda', 'cpu'], help='CPU/CUDA device option')
    parser.add_argument('--fps', default=30, type=int, help='specify fps value of the output video when using rawframes to generate file')
    parser.add_argument('--font-scale', default=12, type=float, help='font scale of the text in the output video')
    parser.add_argument('--font-color', default='white',help='font color of the text in output video')
    parser.add_argument('--target-resolution', nargs=2, default=None, type=int, help='target resolution (width, height) for resizing the '
                       'frames when using a video as input. If either dimension is set to -1, the frames are resized by keeping the existing '
                       'aspect ratio')
    parser.add_argument('--out-filename', default=None, help='output filename')
    
    return parser.parse_args(argument)
    
output_dirpath='D:/results/ucf101'
config_fpath='../config/slowfast_r50_8xb8-4x16x1-256e_kinetics400-rgb.py'
checkpoint_fpath=f'{output_dirpath}/mmaction2-slowfast/demo/slowfast_r50_8xb8-4x16x1-256e_kinetics400.pth' # actually slowfast_r50_8xb8-4x16x1-256e_kinetics400-rgb_20220901-701b0f6f.pth
output_fpath=f'{output_dirpath}/mmaction2-slowfast/demo/output_notebook.mp4'

argument=f""" {config_fpath} {checkpoint_fpath} {output_dirpath}/demo.mp4
{output_dirpath}/label_map_k400.txt  --out-filename {output_fpath} --font-scale 12  --device cpu 
"""
args=parse_args(argument=argument.split())
cfg=Config.fromfile(args.config)
print(f"{args.video=}, {os.path.isfile(args.video)}")

args.video='D:/results/ucf101/demo.mp4', True


Build and load parameter values of [model](https://github.com/open-mmlab/mmaction2/blob/main/demo/demo.py) using function [`model=init_recognizer(cfg, args.checkpoint, device=args.device)`](https://github.com/open-mmlab/mmaction2/blob/main/mmaction/apis/inference.py#L20)

In [4]:
device=torch.device('cpu')
# model=init_recognizer(cfg, args.checkpoint, device=args.device)
print(f"{cfg.model.keys()=}")
recognizer=Recognizer3D(backbone=cfg.model.backbone, cls_head=cfg.model.cls_head, train_cfg=None, test_cfg=None,
                data_preprocessor=cfg.model.data_preprocessor)

# ---- Save pytorch model state_dict -----
# import torch
# torch.save({'model': model.state_dict(),
#             'backbone': model.backbone.state_dict(),
#             'cls_head': model.cls_head.state_dict(),
#             'data_preprocessor': model.data_preprocessor}, os.path.join(output_dirpath, 'mmaction2-slowfast/demo/slowfast_r50_8xb8-4x16x1-256e_kinetics400_torch.pth'))
# fpath=os.path.join(output_dirpath, 'mmaction2-slowfast/demo/slowfast_r50_8xb8-4x16x1-256e_kinetics400_torch.pth')
# checkpoint=torch.load(fpath, map_location='cpu', weights_only=False)
# print(f"{checkpoint.keys()=}")
orig_checkpoint=torch.load(args.checkpoint, map_location='cpu', weights_only=False)
print(f"{orig_checkpoint.keys()=}")
recognizer.load_state_dict(orig_checkpoint['state_dict'])
recognizer.eval()

cfg.model.keys()=dict_keys(['type', 'backbone', 'cls_head', 'data_preprocessor'])
In resnet3d_slowfast.ResNet3dSlowFast.__init__ slow_pathway={'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': True, 'conv1_kernel': (1, 7, 7), 'dilations': (1, 1, 1, 1), 'conv1_stride_t': 1, 'pool1_stride_t': 1, 'inflate': (0, 0, 1, 1), 'norm_eval': False, 'speed_ratio': 8, 'channel_ratio': 8} 
In resnet3d_slowfast.ResNet3dSlowFast.__init__ fast_pathway={'type': 'resnet3d', 'depth': 50, 'pretrained': None, 'lateral': False, 'base_channels': 8, 'conv1_kernel': (5, 7, 7), 'conv1_stride_t': 1, 'pool1_stride_t': 1, 'norm_eval': False} 
In ResNet3dPathway._calculate_lateral_inplanes: depth=50, expansion=4, base_channels=64
stage 0 ----------
	planes=64, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 1 ----------
	planes=256, self.lateral=True, self.lateral_activate[i]=1, self.lateral_inv=False
stage 2 ----------
	planes=512, self.lateral=True, self.lateral_activate[i

Recognizer3D(
  (data_preprocessor): ActionDataPreprocessor()
  (backbone): ResNet3dSlowFast(
    (slow_path): ResNet3dPathway(
      (conv1): ConvModule(
        (conv): Conv3d(3, 64, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3), bias=False)
        (bn): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activation): ReLU(inplace=True)
      )
      (max_pool): MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1), dilation=1, ceil_mode=False)
      (pool2): MaxPool3d(kernel_size=(2, 1, 1), stride=(2, 1, 1), padding=0, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck3d(
          (conv1): ConvModule(
            (conv): Conv3d(80, 64, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
            (bn): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (activation): ReLU(inplace=True)
          )
          (conv2): ConvModule(
            (conv

In [5]:
cfg.test_pipeline

[{'type': 'DecordInit', 'io_backend': 'disk'},
 {'type': 'SampleFrames',
  'clip_len': 32,
  'frame_interval': 2,
  'num_clips': 10,
  'test_mode': True},
 {'type': 'DecordDecode'},
 {'type': 'Resize', 'scale': (-1, 256)},
 {'type': 'ThreeCrop', 'crop_size': 256},
 {'type': 'FormatShape', 'input_format': 'NCTHW'},
 {'type': 'PackActionInputs'}]